# Peak Season Model Drift Detection - Vantara Commerce

This notebook demonstrates how Briefcase AI solves the critical problem of model performance degradation during peak shopping seasons. When model updates cause performance regressions during Black Friday/Cyber Monday, traditional troubleshooting takes 2-5 days. Briefcase AI enables instant root-cause analysis.

## Problem Context

Every Q4, Vantara Commerce rapidly deploys updated AI models to handle traffic spikes:
- **Search Ranking**: Optimized models for peak query volumes
- **Product Recommendations**: Enhanced models for holiday shopping patterns
- **Fraud Detection**: Tuned models for increased transaction volumes

**The Challenge**: When new model versions cause performance regressions mid-season, there's no quick way to:
- Identify which model version caused the issue
- Reconstruct what the model was processing during problem periods
- Determine the exact rollback target

## The Briefcase AI Solution

Briefcase AI captures complete model execution context, enabling instant analysis:
- **Model Version Tracking**: Every decision records exact version used
- **Performance Monitoring**: Real-time confidence scores and fallback rates
- **Context Preservation**: Full input/output available for any decision
- **Instant Root Cause**: Query by version and timestamp for immediate insights

## Setup and Initialization

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random

# Add shared module to path
sys.path.append(os.path.join('..', 'shared'))

# Import our demo modules
import backend
from backend import briefcase, COMPANY

# Set deterministic random seed
random.seed(42)

print(f"Peak Season Model Drift Detection Demo")
print(f"Company: {COMPANY['name']}")
print(f"Industry: {COMPANY['industry']}")
print(f"Peak Season Challenge: Q4 model deployments with performance risk")

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase.init()
    print("SUCCESS: Briefcase AI SDK initialized")
except Exception as e:
    print(f"INFO: Using mock implementation ({e})")

# Get backend for storage
backend_instance = backend.get_backend()
print("SUCCESS: In-memory SQLite backend configured")

## Peak Season Configuration

Vantara Commerce deploys AI models in two waves during Q4:
- **Wave 1 (Pre-BFCM)**: Nov 1-25, stable baseline versions
- **Wave 2 (Post-BFCM)**: Nov 29-Dec 5, optimized holiday versions

Let's examine the model configuration for each wave.

In [ ]:
# Load wave configurations from the example
from example import WAVE_1_CONFIG, WAVE_2_CONFIG

# Display wave configurations
print("WAVE 1 CONFIGURATION (Pre-BFCM):")
print(f"  Period: {WAVE_1_CONFIG['period']}")
print(f"  Dates: {WAVE_1_CONFIG['start_date'].strftime('%b %d')} - {WAVE_1_CONFIG['end_date'].strftime('%b %d')}")
print(f"  Search Ranking Model: {WAVE_1_CONFIG['search_ranking']['model_version']}")
print(f"    - Expected confidence: {WAVE_1_CONFIG['search_ranking']['confidence_mean']:.3f}")
print(f"  Product Recs Model: {WAVE_1_CONFIG['product_recommendations']['model_version']}")
print(f"    - Expected CTR: {WAVE_1_CONFIG['product_recommendations']['ctr_prediction_mean']:.3f}")

print("\nWAVE 2 CONFIGURATION (Post-BFCM):")
print(f"  Period: {WAVE_2_CONFIG['period']}")
print(f"  Dates: {WAVE_2_CONFIG['start_date'].strftime('%b %d')} - {WAVE_2_CONFIG['end_date'].strftime('%b %d')}")
print(f"  Search Ranking Model: {WAVE_2_CONFIG['search_ranking']['model_version']}")
print(f"    - Expected confidence: {WAVE_2_CONFIG['search_ranking']['confidence_mean']:.3f}")
print(f"  Product Recs Model: {WAVE_2_CONFIG['product_recommendations']['model_version']}")
print(f"    - Expected CTR: {WAVE_2_CONFIG['product_recommendations']['ctr_prediction_mean']:.3f}")

## Expected vs Actual Performance

Let's visualize the expected performance differences between the two waves.

In [ ]:
# Create performance comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Search Ranking Confidence Comparison
waves = ['Pre-BFCM\n(v8.2.1-stable)', 'Post-BFCM\n(v8.3.0-bfcm)']
confidence_scores = [WAVE_1_CONFIG['search_ranking']['confidence_mean'], 
                    WAVE_2_CONFIG['search_ranking']['confidence_mean']]

bars1 = ax1.bar(waves, confidence_scores, color=['green', 'red'], alpha=0.7)
ax1.set_title('Search Ranking Model Confidence', fontsize=14, fontweight='bold')
ax1.set_ylabel('Average Confidence Score')
ax1.set_ylim(0, 1.0)

# Add value labels on bars
for bar, score in zip(bars1, confidence_scores):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# Product Recommendations CTR Comparison
ctr_scores = [WAVE_1_CONFIG['product_recommendations']['ctr_prediction_mean'], 
              WAVE_2_CONFIG['product_recommendations']['ctr_prediction_mean']]

bars2 = ax2.bar(waves, ctr_scores, color=['green', 'red'], alpha=0.7)
ax2.set_title('Product Recommendations CTR Prediction', fontsize=14, fontweight='bold')
ax2.set_ylabel('Average CTR Prediction')
ax2.set_ylim(0, max(ctr_scores) * 1.2)

# Add value labels on bars
for bar, score in zip(bars2, ctr_scores):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.002,
             f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate performance degradation
confidence_degradation = ((WAVE_1_CONFIG['search_ranking']['confidence_mean'] - 
                          WAVE_2_CONFIG['search_ranking']['confidence_mean']) / 
                         WAVE_1_CONFIG['search_ranking']['confidence_mean']) * 100

ctr_degradation = ((WAVE_1_CONFIG['product_recommendations']['ctr_prediction_mean'] - 
                   WAVE_2_CONFIG['product_recommendations']['ctr_prediction_mean']) / 
                  WAVE_1_CONFIG['product_recommendations']['ctr_prediction_mean']) * 100

print(f"\nPERFORMNCE IMPACT ANALYSIS:")
print(f"Search Ranking Confidence: {confidence_degradation:.1f}% degradation")
print(f"Product Recommendations CTR: {ctr_degradation:.1f}% degradation")
print(f"\nThis represents significant business impact requiring immediate investigation.")

## Running the Drift Detection Simulation

Now let's simulate the actual peak season deployment and capture the model performance data.

In [ ]:
# Import and run the drift detection simulation
from example import simulate_peak_season_decisions

print("Running peak season drift detection simulation...")
peak_decisions = simulate_peak_season_decisions()

print(f"SUCCESS: Generated {len(peak_decisions)} decisions across both waves")

# Store all decisions in the backend
stored_decision_ids = []
for decision in peak_decisions:
    decision_id = backend_instance.store_decision(decision)
    stored_decision_ids.append(decision_id)

print(f"SUCCESS: {len(stored_decision_ids)} decision records stored in audit trail")

## Analyzing Model Performance by Wave

Let's extract and analyze the captured decision data to detect performance drift.

In [ ]:
# Extract decision data for analysis
decision_data = []

for decision in peak_decisions:
    # Extract inputs and outputs
    team_name = decision.inputs[0].value
    agent_name = decision.inputs[1].value
    vendor = decision.inputs[2].value
    model_name = decision.inputs[3].value
    model_version = decision.inputs[4].value
    query_context = decision.inputs[5].value
    input_tokens = int(decision.inputs[6].value)
    output_tokens = int(decision.inputs[7].value)
    decision_timestamp = decision.inputs[8].value
    wave = decision.inputs[9].value
    
    # Extract outputs
    confidence_score = float(decision.outputs[0].value) if decision.outputs[0].value != 'None' else None
    ctr_prediction = float(decision.outputs[1].value) if decision.outputs[1].value != 'None' else None
    result = decision.outputs[2].value
    
    decision_data.append({
        'team_name': team_name,
        'agent_name': agent_name,
        'vendor': vendor,
        'model_name': model_name,
        'model_version': model_version,
        'query_context': query_context,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'decision_timestamp': decision_timestamp,
        'wave': wave,
        'confidence_score': confidence_score,
        'ctr_prediction': ctr_prediction,
        'result': result,
        'decision_id': decision.decision_id
    })

decisions_df = pd.DataFrame(decision_data)

print(f"DECISION ANALYSIS SUMMARY:")
print(f"Total decisions captured: {len(decisions_df)}")
print(f"Waves represented: {sorted(decisions_df['wave'].unique())}")
print(f"Model versions tracked: {sorted(decisions_df['model_version'].unique())}")
print(f"Teams monitored: {sorted(decisions_df['team_name'].unique())}")

## Performance Degradation Detection

Now let's analyze the actual performance differences between waves to detect model drift.

In [ ]:
# Analyze search ranking performance by wave
search_data = decisions_df[decisions_df['team_name'] == 'search-ranking']
recs_data = decisions_df[decisions_df['team_name'] == 'product-recommendations']

# Search ranking analysis
search_pre = search_data[search_data['wave'] == 'pre_bfcm']
search_post = search_data[search_data['wave'] == 'post_bfcm']

pre_confidence_mean = search_pre['confidence_score'].mean()
post_confidence_mean = search_post['confidence_score'].mean()
confidence_degradation_actual = ((pre_confidence_mean - post_confidence_mean) / pre_confidence_mean) * 100

# Calculate fallback rates
pre_fallback_rate = (search_pre['result'] == 'fallback_to_rules').mean() * 100
post_fallback_rate = (search_post['result'] == 'fallback_to_rules').mean() * 100

print("SEARCH RANKING — MODEL VERSION CHANGE DETECTED:")
print(f"  Pre-BFCM  ({search_pre['model_version'].iloc[0]}): mean confidence {pre_confidence_mean:.3f}, fallback rate {pre_fallback_rate:.0f}%")
print(f"  Post-BFCM ({search_post['model_version'].iloc[0]}): mean confidence {post_confidence_mean:.3f}, fallback rate {post_fallback_rate:.0f}%")
print(f"  Δ confidence: {post_confidence_mean - pre_confidence_mean:.3f} ({confidence_degradation_actual:.0f}% degradation)")
print(f"  Root cause: model_version changed from {search_pre['model_version'].iloc[0]} → {search_post['model_version'].iloc[0]} on Nov 28")

# Product recommendations analysis
recs_pre = recs_data[recs_data['wave'] == 'pre_bfcm']
recs_post = recs_data[recs_data['wave'] == 'post_bfcm']

pre_ctr_mean = recs_pre['ctr_prediction'].mean()
post_ctr_mean = recs_post['ctr_prediction'].mean()
ctr_degradation_actual = ((pre_ctr_mean - post_ctr_mean) / pre_ctr_mean) * 100

print(f"\nPRODUCT RECOMMENDATIONS — MODEL VERSION CHANGE DETECTED:")
print(f"  Pre-BFCM  ({recs_pre['model_version'].iloc[0]}): mean predicted CTR {pre_ctr_mean:.4f}")
print(f"  Post-BFCM ({recs_post['model_version'].iloc[0]}): mean predicted CTR {post_ctr_mean:.4f}")
print(f"  Δ CTR prediction: {post_ctr_mean - pre_ctr_mean:.4f} ({ctr_degradation_actual:.0f}% degradation)")
print(f"  Root cause: model_version changed from {recs_pre['model_version'].iloc[0]} → {recs_post['model_version'].iloc[0]} on Nov 28")

## Visualizing Performance Drift

Let's create comprehensive visualizations showing the performance degradation.

In [ ]:
# Create detailed performance analysis charts
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Vantara Commerce Peak Season Model Drift Analysis', fontsize=16, fontweight='bold')

# 1. Search Ranking Confidence by Wave
search_confidence_data = [search_pre['confidence_score'].tolist(), search_post['confidence_score'].tolist()]
search_labels = ['Pre-BFCM\n(v8.2.1-stable)', 'Post-BFCM\n(v8.3.0-bfcm)']

box1 = ax1.boxplot(search_confidence_data, labels=search_labels, patch_artist=True)
box1['boxes'][0].set_facecolor('lightgreen')
box1['boxes'][1].set_facecolor('lightcoral')
ax1.set_title('Search Ranking Confidence Distribution')
ax1.set_ylabel('Confidence Score')
ax1.grid(True, alpha=0.3)

# 2. Product Recommendations CTR by Wave  
recs_ctr_data = [recs_pre['ctr_prediction'].tolist(), recs_post['ctr_prediction'].tolist()]
recs_labels = ['Pre-BFCM\n(recs-v12.0)', 'Post-BFCM\n(recs-v12.1-bfcm)']

box2 = ax2.boxplot(recs_ctr_data, labels=recs_labels, patch_artist=True)
box2['boxes'][0].set_facecolor('lightgreen')
box2['boxes'][1].set_facecolor('lightcoral')
ax2.set_title('Product Recommendations CTR Distribution')
ax2.set_ylabel('CTR Prediction')
ax2.grid(True, alpha=0.3)

# 3. Fallback Rate Comparison
fallback_rates = [pre_fallback_rate, post_fallback_rate]
fallback_labels = ['Pre-BFCM', 'Post-BFCM']

bars3 = ax3.bar(fallback_labels, fallback_rates, color=['green', 'red'], alpha=0.7)
ax3.set_title('Search Ranking Fallback Rate')
ax3.set_ylabel('Fallback Rate (%)')
ax3.set_ylim(0, max(fallback_rates) * 1.2)

for bar, rate in zip(bars3, fallback_rates):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{rate:.0f}%', ha='center', va='bottom', fontweight='bold')

# 4. Performance Impact Summary
metrics = ['Search\nConfidence', 'Recommendation\nCTR']
degradations = [confidence_degradation_actual, ctr_degradation_actual]

bars4 = ax4.bar(metrics, degradations, color='red', alpha=0.7)
ax4.set_title('Performance Degradation Summary')
ax4.set_ylabel('Performance Degradation (%)')
ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)

for bar, deg in zip(bars4, degradations):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height - 2,
             f'{deg:.0f}%', ha='center', va='top', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## Root Cause Analysis with Decision Context

Let's examine specific problematic decisions to understand what caused the performance degradation.

In [ ]:
# Find the most problematic decisions in the post-BFCM wave
print("ROOT CAUSE ANALYSIS - EXAMINING PROBLEMATIC DECISIONS:")
print("=" * 60)

# Find search ranking fallback decisions
search_fallbacks = search_post[search_post['result'] == 'fallback_to_rules']
if not search_fallbacks.empty:
    print(f"\nSEARCH RANKING FALLBACKS ({len(search_fallbacks)} decisions):")
    for idx, decision in search_fallbacks.iterrows():
        print(f"\nDecision ID: {decision['decision_id']}")
        print(f"  Model: {decision['model_version']} (confidence: {decision['confidence_score']:.3f})")
        print(f"  Query: {decision['query_context'][:80]}...")
        print(f"  Result: Fallback to rules-based system (model confidence too low)")
        print(f"  Impact: Customer sees suboptimal search results")

# Find lowest CTR predictions in recommendations
low_ctr_threshold = recs_post['ctr_prediction'].quantile(0.3)
low_ctr_decisions = recs_post[recs_post['ctr_prediction'] <= low_ctr_threshold]

print(f"\nLOW CTR PREDICTIONS (bottom 30%, threshold ≤ {low_ctr_threshold:.4f}):")
for idx, decision in low_ctr_decisions.head(2).iterrows():
    print(f"\nDecision ID: {decision['decision_id']}")
    print(f"  Model: {decision['model_version']} (predicted CTR: {decision['ctr_prediction']:.4f})")
    print(f"  Context: {decision['query_context'][:80]}...")
    print(f"  Impact: Poor recommendation quality, reduced click-through")

# Compare to historical good performance
print(f"\nHISTORICAL BASELINE (Pre-BFCM Performance):")
good_search = search_pre.iloc[0]
good_recs = recs_pre.iloc[0]

print(f"\nGood Search Decision (ID: {good_search['decision_id']}):")
print(f"  Model: {good_search['model_version']} (confidence: {good_search['confidence_score']:.3f})")
print(f"  Result: Served successfully")

print(f"\nGood Recommendation Decision (ID: {good_recs['decision_id']}):")
print(f"  Model: {good_recs['model_version']} (predicted CTR: {good_recs['ctr_prediction']:.4f})")
print(f"  Result: High engagement expected")

## Traditional vs Briefcase AI Comparison

Let's demonstrate the time savings and accuracy benefits of using Briefcase AI.

In [ ]:
print("INVESTIGATION METHOD COMPARISON:")
print("=" * 50)

print("\nWITHOUT BRIEFCASE AI:")
print("  Time to identify root cause: 2–5 days")
print("  Process:")
print("    1. Manually correlate logs across 2 AI vendors")
print("    2. Contact vendor support for deployment timelines")
print("    3. Reconstruct model version changes from git/deployment logs")
print("    4. Attempt to correlate performance drops with deployments")
print("    5. Limited context reconstruction (logs may be incomplete)")
print("  Evidence quality: Reconstructed, potentially incomplete")
print("  Risk: Continued customer impact during investigation")

print("\nWITH BRIEFCASE AI:")
print("  Time to identify root cause: < 1 minute")
print("  Process:")
print("    1. Query decisions by model_version and decision_timestamp")
print("    2. Compare performance metrics between versions")
print("    3. Access full decision context for any problematic case")
print("  Evidence quality: Immutable decision traces captured at execution time")
print("  Rollback target: Exact version identification")

print("\nKEY BENEFITS:")
print(f"  ⚡ Speed: 2880x faster root cause identification (5 days → 1 minute)")
print(f"  🎯 Precision: Exact model version pinpointing")
print(f"  📊 Context: Complete input/output preservation")
print(f"  🔒 Reliability: Immutable contemporaneous records")
print(f"  💰 Impact: Minimized revenue loss during peak season")

## Audit Trail Verification

Let's verify that all peak season decisions are properly stored and retrievable.

In [ ]:
# Verify audit trail integrity
print("AUDIT TRAIL VERIFICATION:")
print("=" * 40)

verification_results = []
for decision_id in stored_decision_ids:
    retrieved_decision = backend_instance.load_decision(decision_id)
    if retrieved_decision:
        model_version = retrieved_decision.inputs[4].value
        team_name = retrieved_decision.inputs[0].value
        verification_results.append({
            'decision_id': decision_id,
            'team': team_name,
            'model_version': model_version,
            'retrieved': True
        })
    else:
        verification_results.append({
            'decision_id': decision_id,
            'team': 'UNKNOWN',
            'model_version': 'UNKNOWN',
            'retrieved': False
        })

verification_df = pd.DataFrame(verification_results)
success_rate = verification_df['retrieved'].mean() * 100

print(f"Verification Results:")
print(f"  Total records tested: {len(verification_df)}")
print(f"  Successfully retrieved: {verification_df['retrieved'].sum()}")
print(f"  Success rate: {success_rate:.1f}%")

if success_rate == 100:
    print(f"  [SUCCESS] All peak season decisions retrievable from audit trail")
    print(f"  [SUCCESS] Complete model version history preserved")
    print(f"  [SUCCESS] Root cause analysis data permanently available")
else:
    print(f"  [WARNING] Some records could not be retrieved")

## Generate Complete Drift Report

Finally, let's generate the comprehensive drift analysis report.

In [ ]:
# Generate the full drift report
from example import print_drift_report

print("\n" + "=" * 80)
print("COMPLETE PEAK SEASON DRIFT ANALYSIS REPORT")
print("=" * 80)

print_drift_report(peak_decisions, backend_instance)

## Key Takeaways

This walkthrough demonstrated critical capabilities for peak season AI operations:

### 1. Instant Model Version Tracking
- **Complete Version History**: Every decision records exact model version
- **Deployment Correlation**: Immediate identification of version change impacts
- **No Infrastructure Access**: Works without vendor dashboard or cloud credentials

### 2. Performance Degradation Detection
- **Real-time Metrics**: Confidence scores, CTR predictions, fallback rates
- **Statistical Analysis**: Quantified performance differences between versions
- **Business Impact**: Direct correlation to customer experience metrics

### 3. Instant Root Cause Analysis
- **Sub-minute Resolution**: 2880x faster than traditional log correlation
- **Precise Attribution**: Exact model version causing performance issues
- **Full Context**: Complete input/output preservation for debugging

### 4. Operational Benefits
- **Revenue Protection**: Rapid resolution minimizes peak season sales impact
- **Confidence in Deployments**: Immediate performance feedback
- **Regulatory Compliance**: Complete audit trail for model governance

### 5. Peak Season Results
- **42% confidence degradation** in search ranking detected immediately
- **58% CTR prediction degradation** in recommendations identified
- **Exact rollback targets** provided: v8.2.1-stable and recs-v12.0
- **Complete evidence trail** preserved for post-incident analysis

## Next Steps

- **[Agent Discovery](../01_agent_discovery/)**: Identify all models requiring drift monitoring
- **[Cost Attribution](../02_cost_attribution/)**: Track financial impact of model rollbacks
- **[Governance Reporting](../04_governance_report/)**: Include drift analysis in compliance documentation

This drift detection capability ensures rapid response to performance issues during critical business periods, protecting revenue and customer experience.